# Baseline and masked model training

## Setup

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchmetrics
from torch.utils.data import TensorDataset, DataLoader

from src.dataset import load_processed_tracks_dataset, split_tracks_by_song
from src.model import BaselineModel, MaskedModel
from src.transform import convert_tracks_to_input_and_candidate_mask_arrays

sizes = [500, 1000, 2000, 4000, 8000, 16000]
n_val, n_test = 500, 500
n_epochs = 15
learning_rate = 0.1

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

path = "../data/processed/tab_tracks.parquet"
df_files = pd.read_parquet(path, columns=["file"])

device

## Fixed validation and test sets

In [ ]:
_, val_files, test_files = split_tracks_by_song(df_files, n_train=1, n_val=n_val, n_test=n_test, seed=1)

df_heldout = load_processed_tracks_dataset(path, files=val_files + test_files)
df_val = df_heldout[df_heldout["file"].isin(val_files)]
df_test = df_heldout[df_heldout["file"].isin(test_files)]

X_val, M_val, y_val = convert_tracks_to_input_and_candidate_mask_arrays(df_val["tab_frames"])
X_test, M_test, y_test = convert_tracks_to_input_and_candidate_mask_arrays(df_test["tab_frames"])

X_val_t, M_val_t, y_val_t = torch.from_numpy(X_val), torch.from_numpy(M_val), torch.from_numpy(y_val)
X_test_t, M_test_t, y_test_t = torch.from_numpy(X_test), torch.from_numpy(M_test), torch.from_numpy(y_test)

val_loader = DataLoader(TensorDataset(X_val_t, M_val_t, y_val_t), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test_t, M_test_t, y_test_t), batch_size=32)

len(y_val), len(y_test)

## Train and evaluate

In [ ]:
exact_match = torchmetrics.classification.MultilabelExactMatch(num_labels=150).to(device)


def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []

    with torch.no_grad():
        for X_batch, mask_batch, y_batch in data_loader:
            X_batch = X_batch.to(device).float()
            mask_batch = mask_batch.to(device).float()
            y_batch = y_batch.to(device).float()
            metrics.append(metric_fn(model(X_batch, mask_batch), y_batch))

    return aggregate_fn(torch.stack(metrics))


def train(model, optimizer, criterion, train_loader, val_loader, n_epochs):
    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.

        for X_batch, mask_batch, y_batch in train_loader:
            X_batch = X_batch.to(device).float()
            mask_batch = mask_batch.to(device).float()
            y_batch = y_batch.to(device).float()

            loss = criterion(model(X_batch, mask_batch), y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, criterion).item()
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

        print(f"epoch {epoch + 1} / {n_epochs}, train {train_loss:.4f}, val {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return history


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()

    with torch.no_grad():
        for X_batch, mask_batch, y_batch in data_loader:
            X_batch = X_batch.to(device).float()
            mask_batch = mask_batch.to(device).float()
            metric.update(model(X_batch, mask_batch), y_batch.to(device).long())

    return metric.compute().item()


def score_by_category(model, X_t, M_t, y_t, full_loader):
    active_count = y_t.sum(dim=1)

    def loader_for(row_mask):
        return DataLoader(TensorDataset(X_t[row_mask], M_t[row_mask], y_t[row_mask]), batch_size=32)

    return {
        "overall": evaluate_tm(model, full_loader, exact_match),
        "one_note": evaluate_tm(model, loader_for(active_count == 1), exact_match),
        "two_notes": evaluate_tm(model, loader_for(active_count == 2), exact_match),
        "three_plus": evaluate_tm(model, loader_for(active_count > 2), exact_match),
    }


def error_breakdown(model, data_loader, M_t, y_t):
    model.eval()
    batches = []

    with torch.no_grad():
        for X_batch, mask_batch, _ in data_loader:
            logits = model(X_batch.to(device).float(), mask_batch.to(device).float())
            batches.append((torch.sigmoid(logits) > 0.5).cpu().to(torch.uint8))

    preds = torch.cat(batches)

    return {
        "n_frames": len(y_t),
        "n_wrong": (preds != y_t).any(dim=1).sum().item(),
        "n_pitch_invalid": ((preds == 1) & (M_t == 0)).any(dim=1).sum().item(),
    }

## Test frames for the article plots

In [ ]:
out_dir = Path("../data/models")
out_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(0)
sample_idx = np.sort(rng.choice(len(y_test), size=2000, replace=False))

np.savez_compressed(
    out_dir / "test_examples.npz",
    X=X_test[sample_idx],
    M=M_test[sample_idx],
    y=y_test[sample_idx],
)

## One run per training size

In [ ]:
for n_train in sizes:
    print(f"\n=== {n_train} tracks ===")

    train_files, _, _ = split_tracks_by_song(df_files, n_train=n_train, n_val=n_val, n_test=n_test, seed=1)
    df_train = load_processed_tracks_dataset(path, files=train_files)
    X_train, M_train, y_train = convert_tracks_to_input_and_candidate_mask_arrays(df_train["tab_frames"])

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(M_train), torch.from_numpy(y_train)),
        batch_size=32,
        shuffle=True,
    )

    runs = {}
    for name, Model in (("baseline", BaselineModel), ("masked", MaskedModel)):
        print(f"{name}")
        torch.manual_seed(42)
        model = Model(n_input=728).to(device)
        history = train(
            model,
            torch.optim.SGD(model.parameters(), lr=learning_rate),
            nn.BCEWithLogitsLoss(),
            train_loader,
            val_loader,
            n_epochs=n_epochs,
        )
        torch.save(model.state_dict(), out_dir / f"{name}_{n_train}.pt")
        runs[name] = (model, history)

    metrics = {
        "n_train_tracks": len(df_train),
        "n_train_frames": len(y_train),
        "n_val_tracks": len(df_val),
        "n_test_tracks": len(df_test),
        "n_epochs": n_epochs,
        "learning_rate": learning_rate,
        "history": {name: history for name, (_, history) in runs.items()},
        "exact_match": {
            name: {
                "val": score_by_category(model, X_val_t, M_val_t, y_val_t, val_loader),
                "test": score_by_category(model, X_test_t, M_test_t, y_test_t, test_loader),
            }
            for name, (model, _) in runs.items()
        },
        "test_errors": {
            name: error_breakdown(model, test_loader, M_test_t, y_test_t)
            for name, (model, _) in runs.items()
        },
    }
    (out_dir / f"metrics_{n_train}.json").write_text(json.dumps(metrics, indent=2))

    del X_train, M_train, y_train, df_train, train_loader, runs, model
    gc.collect()